# Generate Synthetic Internal Waves: Training

Curated from the archived research notebook `3. Internal waves(training).ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

Full preprocessing requires input files and an `internal_waves` module that
were not included in the available collection. See `docs/reproduction.md`.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='preprocessing')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import xarray as xr
import numpy as np
import scipy
import cmocean as cmo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from glob2 import glob
import dask.array
from swath_rossby_wave import inversion
from tqdm import tqdm
from swath_rossby_wave import skill_matrix, build_h_matrix2, build_hswath_matrix2, inversion, make_error_over_time
import matplotlib.patches as mpatches
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from internal_waves import SpectralDomain, calc_gm_wavenumber_spectra, calc_gm_wavenumber_spectra_on_domain, make_synthetic_field, abel_integral


In [ ]:
# Parameters for the domain and grid
N = 2**8  # Number of points in the domain
dx = 8  # Resolution in km
lon0 = -135.0  # Longitude of lower left corner of the grid
lat0 = 25.0  # Latitude of lower left corner of the grid
seed=13579


In [ ]:
# Create domain with specified N and dx
domain = SpectralDomain(N=N, dx=dx)

# Calculate spectra on the domain
SSH2D = calc_gm_wavenumber_spectra_on_domain(domain.kappa)

# Scale SSH to m^2/cpkm
scaling_factor = 2 * np.pi * 1e-3

# Plotting the results
SSH2D_scaled = SSH2D * domain.geometric_factor * (domain.N**2) / (domain.dx**2)  # convert SSH per unit theta to m^2/cpkm/cpkm
SSH2D_scaled[domain.N//2, domain.N//2] = 0.0


In [ ]:
# Generate synthetic field from 2D SSH
nr = 20 * 117
# make_mask = (domain.kappa > 1.001e-2) & (domain.kappa < 1.002e-2)
ssh_iw = make_synthetic_field(SSH2D_scaled, nr=nr, seed=seed)
lon_i, lat_i = domain.make_latlon_grid(lon0, lat0)

plt.contourf(lon_i, lat_i, ssh_iw[..., 0], levels=30, cmap='RdBu_r')
plt.colorbar()
plt.title('Internal Wave SSH')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


## Save the generated field

This export cell was added to supply the intermediate file consumed by the
projection notebook. It follows the testing notebook schema.

In [ ]:
internal_wave_dataset = xr.Dataset({
    "internal_waves": (["N0", "N1", "nr"], ssh_iw),
    "lon_i": (["N0", "N1"], lon_i),
    "lat_i": (["N0", "N1"], lat_i),
})
internal_wave_dataset.to_netcdf(output_path("new_internal_waves.nc"))
